In [6]:
import tkinter as tk
import random
import threading
import time

In [ ]:
ROWS = 5
COLS = 9
CELL = 80

# Coordinates (row, col)
START = (2, 1)          # green
OUTCOME = (2, 7)        # red
ACTION_COL = 4          # vertical yellow column
ACTION_ROWS = {
    "pass": 0,
    "emergency": 1,
    "dump": 2,
    "quick": 3,
    "high": 4,
}

ACTIONS = list(ACTION_ROWS.keys())

# Volleyball conditions
ROW_OPTIONS = ["front", "back"]
PASS_OPTIONS = ["good", "bad"]
BLOCK_OPTIONS = ["blockers", "noblock"]

In [8]:
class Env:
    def __init__(self):
        self.reset()

    def reset(self):
        # fresh rally
        self.pos = list(START)
        self.steps = 0
        # assign rally conditions (state encoding is hidden but stored here)
        self.conditions = (
            random.choice(ROW_OPTIONS),
            random.choice(PASS_OPTIONS),
            random.choice(BLOCK_OPTIONS),
        )
        self.chosen_action = None
        self.visited_actions = set()
        self.done = False
        return self.observe()

    def observe(self):
        # observation is only the conditions; agent uses value function on grid positions (like original notebook)
        return self.conditions

    def on_action_tile(self, r, c):
        return c == ACTION_COL and r in ACTION_ROWS.values()

    def action_from_rc(self, r, c):
        if c != ACTION_COL: 
            return None
        for name, rr in ACTION_ROWS.items():
            if rr == r: return name
        return None

    def legal_for_conditions(self, action):
        row, pquality, _ = self.conditions
        if pquality == "good":
            if row == "front":
                legal = {"high", "quick", "dump", "pass"}
            else:  # back
                legal = {"high", "quick", "pass"}
        else:  # bad pass
            if row == "front":
                legal = {"emergency", "dump", "pass"}
            else:  # back + bad
                legal = {"emergency", "pass"}  # dump illegal from back row on bad pass per user
        return action in legal

    def step(self, action_move):
        if self.done:
            return self.observe(), 0.0, True, {}

        self.steps += 1
        r, c = self.pos

        # move according to gridworld action
        if action_move == "up":    r = max(0, r - 1)
        if action_move == "down":  r = min(ROWS - 1, r + 1)
        if action_move == "left":  c = max(0, c - 1)
        if action_move == "right": c = min(COLS - 1, c + 1)

        self.pos = [r, c]

        reward = 0.0

        # penalty if taking too long (after limit)
# penalty increases by 0.1 for each step after 10
        if self.steps > 10:
            reward -= 0.1 * (self.steps - 10)

        # if on an action tile
        if self.on_action_tile(r, c):
            a = self.action_from_rc(r, c)
            if a in self.visited_actions:
                # revisiting same action tile counts as multiple decisions -> penalty + end
                reward -= 0.5
                self.done = True
            else:
                self.visited_actions.add(a)
                if self.chosen_action is None:
                    self.chosen_action = a
                    # check legality immediately
                    if not self.legal_for_conditions(a):
                        reward -= 1.0
                        self.done = True

        # if reached outcome tile, resolve rally
        if (r, c) == OUTCOME and not self.done:
            reward += self.resolve_outcome()
            self.done = True

        return self.observe(), reward, self.done, {}

    def resolve_outcome(self):
        # compute rally result based on conditions and chosen_action
        row, pquality, blockers = self.conditions
        a = self.chosen_action

        if a is None:
            return -0.8  # reached outcome without selecting a set

        # base success probabilities (you can tune these)
        probs = {
            "high":      0.65 if pquality == "good" else 0.5,
            "quick":     0.6  if pquality == "good" else 0.35,
            "dump":      0.5  if row == "front" else 0.1,  # back-row dump is illegal earlier; keep low anyway
            "emergency": 0.35 if pquality == "bad" else 0.40,
            "pass":      None,  # handled separately
        }

        if a == "pass":
            return -0.6  # safe fallback

        p = probs[a]
        # blocker effect
        if blockers == "blockers":
            p *= 0.8  # more chance to be blocked
        else:
            p *= 1.05

        return 1.0 if random.random() < p else -1.0


In [9]:
class Agent:
    def __init__(self, env):
        self.env = env
        self.lr = 0.1
        self.gamma = 0.9
        self.exp_rate = 0.1  # exploration on movement
        self.initial_exp_rate = 0.5  # Starting exploration rate
        self.min_exp_rate = 0.1     # Minimum exploration rate
        self.decay_rate = 0.995      # Decay factor per episode
        self.exp_rate = self.initial_exp_rate
        self.episode_count = 0       # Track episodes for decay
        # value function on grid positions (same shape as original)
        self.V = {(r, c): 0.0 for r in range(ROWS) for c in range(COLS)}

    def greedy_action(self, pos):
        # look one step ahead and choose movement that leads to highest V(next)
        r, c = pos
        candidates = {
            "up":    (max(0, r-1), c),
            "down":  (min(ROWS-1, r+1), c),
            "left":  (r, max(0, c-1)),
            "right": (r, min(COLS-1, c+1)),
        }
        best_move = max(candidates.keys(), key=lambda a: self.V[candidates[a]])
        return best_move

    def choose_action(self, pos):
        if random.random() < self.exp_rate:
            return random.choice(["up", "down", "left", "right"])
        return self.greedy_action(pos)

    def td_update(self, s, r, s_next):
        self.V[s] += self.lr * (r + self.gamma * self.V[s_next] - self.V[s])

    def play_episode(self, callback=None, delay_ms=0, greedy=False):
        obs = self.env.reset()
        s = tuple(self.env.pos)
        done = False
        total_reward = 0.0
        while not done:
            move = (self.greedy_action(s) if greedy else self.choose_action(s))
            obs_next, r, done, info = self.env.step(move)
            s_next = tuple(self.env.pos)

            # TD(0) update on positions (same algorithmic spirit as notebook)
            self.td_update(s, r, s_next)

            total_reward += r
            s = s_next

            if callback:
                callback(self, obs_next, move, r, done)

            if delay_ms > 0:
                time.sleep(delay_ms / 1000.0)
        self.episode_count += 1
        self.exp_rate = max(self.min_exp_rate, self.initial_exp_rate * (self.decay_rate ** self.episode_count))
        return total_reward

In [10]:
class App:
    def __init__(self):
        self.env = Env()
        self.agent = Agent(self.env)

        self.root = tk.Tk()
        self.root.title("Volleyball Setter Decision Gridworld")

        self.canvas = tk.Canvas(self.root, width=COLS*CELL, height=ROWS*CELL, bg="white")
        self.canvas.pack()

        self.status = tk.Label(self.root, text="", anchor="w")
        self.status.pack(fill="x")

        ctl = tk.Frame(self.root); ctl.pack(fill="x")

        self.train_btn = tk.Button(ctl, text="Train", command=self.start_training); self.train_btn.pack(side="left", padx=5)
        self.pause_btn = tk.Button(ctl, text="Pause", command=self.pause_training); self.pause_btn.pack(side="left", padx=5)
        self.policy_btn = tk.Button(ctl, text="Show Policy", command=self.show_policy); self.policy_btn.pack(side="left", padx=5)
        tk.Label(ctl, text="Speed (ms)").pack(side="left", padx=(20,4))

        # Episode counter label
        self.ep_label = tk.Label(self.root, text="Episodes: 0", anchor="w")
        self.ep_label.pack(fill="x")

        self.speed = tk.Scale(ctl, from_=0, to=800, orient="horizontal"); self.speed.set(200); self.speed.pack(side="left")

        self.draw_board()
        self.agent_token = None
        self.draw_agent()

        self.running = False

    def pause_training(self):
        self.running = False

    def draw_board(self):
        self.canvas.delete("all")
        for r in range(ROWS):
            for c in range(COLS):
                x1, y1 = c*CELL, r*CELL
                x2, y2 = x1+CELL, y1+CELL
                fill = "white"
                # start
                if (r, c) == START:
                    fill = "#15b300"  # green
                # outcome
                if (r, c) == OUTCOME:
                    fill = "#d92323"  # red
                # actions column
                if c == ACTION_COL and r in ACTION_ROWS.values():
                    fill = "#f0b400"  # yellow
                self.canvas.create_rectangle(x1, y1, x2, y2, fill=fill, outline="black")

        # label action tiles
        for name, rr in ACTION_ROWS.items():
            cx = ACTION_COL*CELL + CELL/2
            cy = rr*CELL + CELL/2
            self.canvas.create_text(cx, cy, text=name.title(), font=("Arial", 12, "bold"))

    def draw_agent(self):
        r, c = self.env.pos
        x1, y1 = c*CELL+10, r*CELL+10
        x2, y2 = x1+CELL-20, y1+CELL-20
        if self.agent_token is None:
            self.agent_token = self.canvas.create_oval(x1, y1, x2, y2, fill="#1f6feb", outline="black")
        else:
            self.canvas.coords(self.agent_token, x1, y1, x2, y2)

    def ui_callback(self, agent, obs, move, reward, done):
        # show current conditions and reward
        cond = self.env.conditions
        chosen = self.env.chosen_action
        self.status.config(text=f"State: {cond} | Chosen action: {chosen} | Move: {move} | Reward: {reward:.2f}")
        self.draw_agent()
        self.root.update_idletasks()
        self.root.update()

    def start_training(self):
        if self.running: return
        self.running = True
        def run():
            ep = 0
            while self.running:
                ep += 1
                self.agent.play_episode(callback=self.ui_callback, delay_ms=self.speed.get(), greedy=False)
                self.ep_label.config(text=f"Episodes: {ep}")
            self.status.config(text="Training paused.")
        threading.Thread(target=run, daemon=True).start()

    def show_policy(self):
        if self.running: return
        def run():
            # greedy runs that do not change learning
            for _ in range(10):
                self.agent.play_episode(callback=self.ui_callback, delay_ms=300, greedy=True)
            self.status.config(text="Policy demo finished.")
        threading.Thread(target=run, daemon=True).start()

    def run(self):
        self.root.mainloop()

In [11]:
if __name__ == "__main__":
    App().run()